In [ ]:
df = spark.read.parquet("data/spark-output/cleaned_reviews")
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df.printSchema()


In [ ]:
from pyspark.sql.functions import count
from pyspark.sql.functions import col

top_reviewers = (
    df.groupBy("user_id")
      .agg(count("*").alias("n_reviews"))
      .orderBy(col("n_reviews").desc())
      .limit(20)
)

top_reviewers.show()


In [ ]:
top_reviewers_pd = top_reviewers.toPandas()


In [ ]:
rev_pd = top_reviewers.toPandas()  # rename if needed

print("rev_pd columns:", list(rev_pd.columns))

# find reviewer id column
ucol = None
for c in ["user_id", "reviewerID", "reviewer_id"]:
    if c in rev_pd.columns:
        ucol = c
        break
if ucol is None:
    ucol = rev_pd.columns[0]

# find count column
ycol = None
for c in ["n_reviews", "count", "cnt", "n"]:
    if c in rev_pd.columns:
        ycol = c
        break
if ycol is None:
    for c in rev_pd.columns:
        if "count" in c.lower() or "review" in c.lower():
            ycol = c
            break
if ycol is None:
    raise ValueError("No count column found in rev_pd.")

rev_pd = rev_pd.sort_values(ycol, ascending=False).head(10).sort_values(ycol)

plt.figure(figsize=(10,4))
plt.barh(rev_pd[ucol], rev_pd[ycol])
plt.title("Top 10 Reviewers by Review Count")
plt.xlabel("Number of Reviews")
plt.ylabel(ucol)
plt.tight_layout()
plt.show()

In [ ]:
from pyspark.sql.functions import countDistinct

reviewer_category_diversity = (
    df.join(top_reviewers, on="user_id")
      .groupBy("user_id")
      .agg(
          count("*").alias("n_reviews"),
          countDistinct("main_category_clean").alias("n_categories")
      )
      .orderBy(col("n_reviews").desc())
)

reviewer_category_diversity.show()


In [ ]:
chosen_category = "Beauty & Personal Care"
df_cat = df.filter(col("main_category_clean") == chosen_category)
df_cat.count()


In [ ]:
from pyspark.sql.functions import rand

df_sim = df_cat.orderBy(rand()).limit(1_000_000)


In [ ]:
from pyspark.sql.functions import concat_ws, lower, regexp_replace

df_sim = df_sim.withColumn(
    "text_all",
    lower(
        regexp_replace(
            concat_ws(" ", col("title"), col("text")),
            "[^a-zA-Z0-9 ]", " "
        )
    )
)


In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, MinHashLSH
from pyspark.ml import Pipeline

tokenizer = Tokenizer(inputCol="text_all", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="features", numFeatures=1<<18)

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF])
model = pipeline.fit(df_sim)
df_feat = model.transform(df_sim)

mh = MinHashLSH(
    inputCol="features",
    outputCol="hashes",
    numHashTables=5
)

mh_model = mh.fit(df_feat)


In [ ]:
similar_pairs = mh_model.approxSimilarityJoin(
    df_feat.select("parent_asin", "user_id", "review_date", "text_all", "features"),
    df_feat.select("parent_asin", "user_id", "review_date", "text_all", "features"),
    0.2,   # distance threshold
    distCol="jaccardDist"
).filter("datasetA.text_all != datasetB.text_all")


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

df_sim = df_sim.withColumn("rid", monotonically_increasing_id())


In [ ]:
from pyspark.sql.functions import col, size
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array

# tokenize + remove stopwords
tokenizer = Tokenizer(inputCol="text_all", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")

tmp = Pipeline(stages=[tokenizer, remover]).fit(df_sim).transform(df_sim)

# drop empty token lists
tmp = tmp.filter(size(col("filtered")) > 0)

# hashing
hashingTF = HashingTF(
    inputCol="filtered",
    outputCol="features",
    numFeatures=1 << 18
)

df_feat = hashingTF.transform(tmp)

# drop zero vectors (CRITICAL for MinHash)
df_feat = df_feat.filter(size(vector_to_array("features")) > 0)


In [ ]:
from pyspark.ml.feature import MinHashLSH

mh = MinHashLSH(
    inputCol="features",
    outputCol="hashes",
    numHashTables=5
)

mh_model = mh.fit(df_feat)


In [ ]:
from pyspark.sql.functions import rand
from pyspark import StorageLevel

df_small_raw = (
    df_sim.select("rid", "text_all")
          .orderBy(rand())
          .limit(100_000)
          .repartition(64)
          .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_small_raw.count()


In [ ]:
from pyspark.sql.functions import size, col
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array

tokenizer = Tokenizer(inputCol="text_all", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="features", numFeatures=1<<18)

pipe = Pipeline(stages=[tokenizer, remover])
tmp = pipe.fit(df_small_raw).transform(df_small_raw)

tmp = tmp.filter(size(col("filtered")) > 0)

df_small = hashingTF.transform(tmp)
df_small = df_small.filter(size(vector_to_array("features")) > 0) \
                   .select("rid","features") \
                   .repartition(64) \
                   .persist(StorageLevel.MEMORY_AND_DISK)

total_count = df_small.count()
total_count


In [ ]:
from pyspark.ml.feature import MinHashLSH

mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
mh_model = mh.fit(df_small)


In [ ]:
from pyspark.sql.functions import explode, countDistinct, col
from pyspark import StorageLevel

df_h = (
    mh_model.transform(df_small)
            .select("rid", explode("hashes").alias("h"))
            .persist(StorageLevel.MEMORY_AND_DISK)
)
_ = df_h.count()

dup_rids = (
    df_h.groupBy("h")
        .agg(countDistinct("rid").alias("n"))
        .filter(col("n") > 1)
        .join(df_h, on="h", how="inner")
        .select("rid")
        .distinct()
)

dup_count = dup_rids.count()
dup_rate = dup_count / total_count
(dup_count, total_count, dup_rate)


In [ ]:
examples = (
    dup_rids.limit(20)
            .join(df_small_raw, on="rid", how="left")
            .select("rid", "text_all")
)
examples.show(10, truncate=120)


In [ ]:
df_small_raw = (
    df_sim.select("rid", "parent_asin", "text_all")
          .orderBy(rand())
          .limit(200_000)
)


In [ ]:
from pyspark.sql.functions import count, col

# count how many other reviews share each hash bucket
bucket_sizes = df_h.groupBy("h").count().filter(col("count") > 1)

# each rid gets credit for each hash bucket it is in that has collisions
rid_collisions = (
    df_h.join(bucket_sizes.select("h"), on="h", how="inner")
        .groupBy("rid")
        .agg(count("*").alias("n_colliding_hashes"))
)

rid_collisions.selectExpr(
    "min(n_colliding_hashes) as min_c",
    "percentile(n_colliding_hashes, 0.5) as p50",
    "percentile(n_colliding_hashes, 0.9) as p90",
    "max(n_colliding_hashes) as max_c"
).show()


In [ ]:
k = 3

dup_count_k = rid_collisions.filter(col("n_colliding_hashes") >= k).count()
dup_rate_k = dup_count_k / total_count
(dup_count_k, total_count, dup_rate_k)


In [ ]:
df_demo = df_small.join(df_small_raw, on="rid", how="left").limit(5000)

# pick one review and find nearest neighbors
key = df_demo.limit(1).collect()[0]["features"]

neighbors = mh_model.approxNearestNeighbors(
    df_demo, key, 5, distCol="jaccardDist"
).select("rid", "jaccardDist", "text_all")

neighbors.show(5, truncate=120)


In [ ]:
from pyspark.sql.functions import count

top5 = (
    df_small_raw.groupBy("parent_asin")
                .agg(count("*").alias("n"))
                .orderBy(col("n").desc())
                .limit(5)
)
top5.show()


In [ ]:
top_asins = [r["parent_asin"] for r in top5.collect()]

rid_to_asin = df_small_raw.select("rid","parent_asin")

dup_by_product = (
    rid_collisions.filter(col("n_colliding_hashes") >= k)
                  .join(rid_to_asin, on="rid", how="left")
                  .filter(col("parent_asin").isin(top_asins))
                  .groupBy("parent_asin")
                  .agg(count("*").alias("dup_reviews"))
)

total_by_product = (
    rid_to_asin.filter(col("parent_asin").isin(top_asins))
               .groupBy("parent_asin")
               .agg(count("*").alias("total_reviews"))
)

result = (
    total_by_product.join(dup_by_product, on="parent_asin", how="left")
                    .fillna(0, subset=["dup_reviews"])
                    .withColumn("dup_rate", col("dup_reviews")/col("total_reviews"))
                    .orderBy(col("dup_rate").desc())
)

result.show(truncate=False)


In [ ]:
result.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "data/spark-output/output/top5_dup_rate_k"
)


In [ ]:
dup_pd = result.toPandas()

print("dup_pd columns:", list(dup_pd.columns))

pcol = "parent_asin" if "parent_asin" in dup_pd.columns else dup_pd.columns[0]
dcol = "dup_rate" if "dup_rate" in dup_pd.columns else None
if dcol is None:
    for c in dup_pd.columns:
        if "dup" in c.lower() and "rate" in c.lower():
            dcol = c
            break
if dcol is None:
    raise ValueError("No duplication rate column found in result.")

dup_pd = dup_pd.sort_values(dcol)

plt.figure(figsize=(8,4))
plt.barh(dup_pd[pcol], dup_pd[dcol])
plt.title("Duplicate Review Rate for Top 5 Products")
plt.xlabel("Duplication Rate")
plt.ylabel(pcol)
plt.tight_layout()
plt.show()

In [ ]:
dup_count_k = rid_collisions.filter(col("n_colliding_hashes") >= k).count()
dup_rate_k = dup_count_k / total_count
(dup_count_k, total_count, dup_rate_k)


In [ ]:
asin_hi = result.orderBy(col("dup_rate").desc()).first()["parent_asin"]

examples = (
    rid_collisions.filter(col("n_colliding_hashes") >= k)
      .join(df_small_raw, on="rid", how="left")
      .filter(col("parent_asin") == asin_hi)
      .select("rid", "parent_asin", "text_all")
      .limit(10)
)

examples.show(truncate=140)
